# RAG Pipeline 

This notebook is the **entry point** for the RAG framework defined in `tragframe.py`.  
It does not implement any indexing, embedding, or retrieval logic — it only orchestrates the framework classes.

**Run order:** execute cells top to bottom once. After the index is built (Cell 4), only Cell 5 needs to be re-run to ask new questions.

## Cell 1 — Imports

In [1]:
import sys
sys.path.insert(0, "..")

import html
from ipywidgets import Textarea, Button, Output, VBox, Layout, HTML
from IPython.display import display
from tragframe import Monitor, VectorDatabase, RAG
from llm_client import OllamaGemmaClient

**What this cell does:**

- `sys.path.insert(0, "..")` — makes the parent directory importable so `tragframe.py` and `llm_client.py` can be found from inside the `notebooks/` folder.
- `tragframe` — the core RAG framework module containing `Monitor`, `VectorDatabase`, and `RAG`.
- `llm_client` — a thin wrapper around `langchain_ollama.OllamaLLM` that exposes a single `invoke(prompt)` method.
- `ipywidgets` — used only in the final cell to render an interactive text input inside the notebook output area.

> **Requirement:** Ollama must be running locally before this notebook is executed.  
> Start it with: `ollama serve`  
> Pull the model if not yet downloaded: `ollama pull gemma3:4b`

## Cell 2 — Configuration

In [2]:
DATA_DIR = "../data"
TOP_K    = 5

**What this cell does:**

Defines the two runtime parameters used throughout the notebook:

| Parameter | Value | Description |
|---|---|---|
| `DATA_DIR` | `../data` | Root directory containing topic subfolders with PDF files. Each subfolder name becomes the topic label for retrieved chunks. |
| `TOP_K` | `5` | Number of document chunks retrieved per query. Higher values provide more context to the LLM but increase prompt length. |

> **To change the data source or tune retrieval behaviour**, edit this cell only — no other cell needs to be modified.

## Cell 3 — Initialize Framework Objects

In [3]:
monitor = Monitor()
db      = VectorDatabase(monitor=monitor)
db.update_database(DATA_DIR)
llm     = OllamaGemmaClient()
rag     = RAG(vector_db=db, llm=llm, monitor=monitor)

Ignoring wrong pointing object 11 0 (offset 0)


## Run retrieval evaluation

In [4]:
EVAL_PATH = "../eval/eval.jsonl"
report = rag.evaluate(EVAL_PATH, top_k=TOP_K)
print("n:", report["n"], "skipped:", report["skipped"])
print("hit@1_mean:", report["hit@1_mean"], "hit@3_mean:", report["hit@3_mean"], "hit@5_mean:", report["hit@5_mean"], "mrr@5_mean:", report["mrr@5_mean"])
if report.get("failures"):
    print("Failures (first 10):", report["failures"][:10])

n: 24 skipped: 0
hit@1_mean: 0.619 hit@3_mean: 0.6667 hit@5_mean: 0.7619 mrr@5_mean: 0.6643
Failures (first 10): [{'id': 'rag3', 'query': 'What is chunking?', 'top_sources': ['Advanced RAG — Improving retrieval using Hypothetical Document Embeddings(HyDE) _ by Plaban Nayak _ AI Planet.pdf', 'Advanced RAG — Improving retrieval using Hypothetical Document Embeddings(HyDE) _ by Plaban Nayak _ AI Planet.pdf', 'Retrieval-Augmented Generation (RAG) _ Pinecone.pdf', 'Retrieval-Augmented Generation (RAG) _ Pinecone.pdf', 'Retrieval-Augmented Generation (RAG) _ Pinecone.pdf'], 'top_chunk_ids': [531, 526, 709, 712, 729]}, {'id': 'rag8', 'query': 'What are embeddings in a RAG pipeline?', 'top_sources': ['Advanced RAG — Improving retrieval using Hypothetical Document Embeddings(HyDE) _ by Plaban Nayak _ AI Planet.pdf', 'Advanced RAG — Improving retrieval using Hypothetical Document Embeddings(HyDE) _ by Plaban Nayak _ AI Planet.pdf', 'Advanced RAG — Improving retrieval using Hypothetical Document 

## Cell 4 — Ask

In [5]:
for _name in ("_text", "_btn", "_out"):
    try:
        globals()[_name].close()
    except KeyError:
        pass

_text = Textarea(placeholder="Type your question here...", layout={"width": "80%", "height": "60px"})
_btn  = Button(description="Ask", button_style="primary")
_out  = HTML(value="<div style='max-height:400px;overflow:auto;white-space:pre;font-family:monospace;'>—</div>")

def _ask(_):
    _btn.disabled = True
    _btn.description = "Thinking..."
    _out.value = "<div style='max-height:400px;overflow:auto;white-space:pre;font-family:monospace;'>Loading...</div>"
    try:
        if _text.value.strip():
            result = rag.retrieve(_text.value.strip(), top_k=TOP_K)
            escaped = html.escape(result)
            _out.value = f"<div style='max-height:400px;overflow:auto;white-space:pre;font-family:monospace;'>{escaped}</div>"
        else:
            _out.value = "<div style='max-height:400px;overflow:auto;white-space:pre;font-family:monospace;'>—</div>"
    finally:
        _btn.disabled = False
        _btn.description = "Ask"

_btn.on_click(_ask)
display(VBox([_text, _btn, _out]))

**What this cell does:**

Renders a minimal interactive query interface directly in the notebook output area:

- **`Textarea`** — a multi-line text field where you type the question. Editable inline, no popup prompt.
- **`Button`** — clicking **Ask** triggers the `_ask` callback.
- **`Output`** — a dedicated output widget that captures the `print()` result from the RAG pipeline and displays it below the button. Using `with _out:` is required in Jupyter to ensure output from a button callback is routed to the correct cell area rather than being lost.
- **`_out.clear_output(wait=True)`** — clears the previous answer before printing the new one so answers do not stack.

**What happens when you click Ask:**

1. The query is passed to `rag.retrieve(query)`.
2. `RAG` runs pre-retrieval guardrails (prompt injection, toxicity, PII, competitor mentions, retrieval quality).
3. If guardrails pass, the top-`K` chunks are formatted into a grounded prompt and sent to the LLM.
4. Post-generation guardrails check the answer for data leakage and uncertain language.
5. The final answer is printed with source citations, or a `Blocked` message with the reason is shown instead.